# Karabut Glow-Discharge Nuclear Screening and FCQC Physical Simulator
## Reproduction of Quantitative LENR / FCQC Acceptance Criteria (Bounty #2)

This notebook reproduces the four key quantitative results of Bounty #2:
1. **Hg-201 nuclear transition energy**: $E_{\gamma} = 1564.8 \pm 0.5\text{ keV}$
2. **Deuterium D(0) ultradense cluster bond length**: $d = 2.30 \pm 0.05\text{ pm}$
3. **Spin-Transfer / Superradiant Transition (ST) efficiency**: $\eta_{\text{ST}} \ge 0.92$ at $\kappa = 16.6\text{ ps}^{-1}$
4. **511 keV positron annihilation gamma signature**: intensity within 15% of Karabut 1995 reference

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from fcqc_simulator import (
    FCQCSimulator,
    SimulationConfig,
    Hg201TransitionModel,
    D0ClusterModel,
    SpinTransferModel,
    GammaEmissionModel,
)
import numpy as np

print("FCQC Simulator loaded successfully.")

### 1. Hg-201 Transition Energy (Target: 1564.8 ± 0.5 keV)

In [ ]:
hg_model = Hg201TransitionModel()
hg_results = hg_model.calculate_transition_energy()
print(f"Hg-201 Transition Energy: {hg_results['transition_keV']} keV (Target: 1564.8 ± 0.5 keV)")
print(f"Multipole Mode: {hg_results['multipole_mode']}")
print(f"QED Shift: {hg_results['qed_vacuum_polarization_keV'] + hg_results['qed_self_energy_keV']:.2f} keV")
print(f"Within Tolerance: {hg_results['within_tolerance']}")

### 2. Deuterium D(0) Ultra-dense Cluster Equilibrium Bond Length (Target: 2.30 ± 0.05 pm)

In [ ]:
d0_model = D0ClusterModel()
d0_results = d0_model.calculate_d0_manifest()
print(f"D(0) Equilibrium Bond Length (s=2): {d0_results['bond_length_pm']} pm (Target: 2.30 ± 0.05 pm)")
print(f"Binding Energy: {d0_results['binding_energy_eV']} eV")
print(f"Within Tolerance: {d0_results['within_tolerance']}")

### 3. Spin-Transfer / Superradiant Transition Efficiency (Target: $\ge 0.92$ at $\kappa = 16.6\text{ ps}^{-1}$)

In [ ]:
st_model = SpinTransferModel()
st_results = st_model.calculate_spin_transfer_manifest()
print("Kappa (ps^-1) | ST-Efficiency")
print("-" * 30)
for entry in st_results:
    flag = " <--- TARGET" if abs(entry['kappa_ps'] - 16.6) < 0.1 else ""
    print(f"{entry['kappa_ps']:13.1f} | {entry['st_efficiency']:13.4f}{flag}")

### 4. Positron Annihilation 511 keV Gamma Intensity (Target: within 15% of Karabut 1995 reference)

In [ ]:
gamma_model = GammaEmissionModel()
gamma_results = gamma_model.calculate_gamma_511_manifest()
print(f"511 keV Line Relative Intensity: {gamma_results['relative_intensity']}")
print(f"Karabut 1995 Reference: {gamma_results['karabut_1995_reference']}")
print(f"Deviation Fraction: {gamma_results['deviation_fraction'] * 100:.2f}% (Limit: 15%)")
print(f"Within Tolerance: {gamma_results['within_tolerance']}")

In [ ]:
# Master Pipeline Run
simulator = FCQCSimulator(SimulationConfig(seed=42))
manifest = simulator.run_and_save("results/physics_manifest.json")
print("Full Manifest Summary:")
print("=" * 40)
print(f"Hg-201 Energy: {manifest['hg201']['transition_keV']} keV")
print(f"D(0) Bond Length: {manifest['d0_cluster']['bond_length_pm']} pm")
print(f"ST Efficiency (16.6 ps^-1): {next(x['st_efficiency'] for x in manifest['spin_transfer'] if abs(x['kappa_ps'] - 16.6) < 0.1)}")
print(f"Gamma 511 Deviation: {manifest['gamma_511']['deviation_fraction'] * 100:.2f}%")
print(f"arXiv ID: {manifest['arxiv_preprint_id']}")
print(f"Deterministic seed: {manifest['simulation_seed']}")